# Layer 1 — Multi-Source Evidence Collection (complete, standalone, v5)

Every function is defined inline (no `regvar` imports needed), matching the standalone nature of the pipeline notebooks.
Needs Layer 0's output for this gene — both the repo layout (`data/outputs/<gene>/...`) and the Layer 0 notebook layout are auto-detected.

### Changelog: What changed across iterations

| Version | Key Changes & Fixes |
|---|---|
| **v3** | Introduced the 3-state missingness model (`observed_positive`, `observed_null`, `absent`), machine-readable reasons, and `MODALITY_META`. |
| **v4** | Decoupled Layer 0 intake execution out of Cell 12 helper definitions; introduced candidate position windowing for GWAS; added baseline HTTP disk caching. |
| **v5** | **Fixed cache poisoning & failure caching:** HTTP 200 with GraphQL `errors` is rejected from disk cache; cached 404 lookup fixed; added `cache_ttl_days` (7 days) and `refresh_cache` flag.<br>**Fixed GWAS window error scoping:** Window errors are now tracked per window (with $\pm 1$ bp padding); a failure in one window no longer mislabels non-hits in healthy windows as `incomplete_region_query`.<br>**Fixed cache config binding & manifest:** `fetch` closure bound to `L1["use_http_cache"]`; manifest records actual `hits`, `misses`, and `oldest_entry_utc`.<br>**Fixed rate-limit delay:** `gnomad_request_delay_s` only sleeps on genuine network calls, executing instantaneously on cache hits.<br>**Restored canonical layer roadmap:** Harmonized text with framework documentation (Layer 0 Intake $\rightarrow$ Layer 1 Evidence $\rightarrow$ Layer 2 Rules $\rightarrow$ Layer 3 AlphaGenome $\rightarrow$ Layer 5 Integration $\rightarrow$ Layer 6 Calibration $\rightarrow$ Layer 7 Output).<br>**Added `strict` mode:** Optional `strict: true` flag in config to fail fast in automated CI pipelines when Layer 0 output is absent. |

**Design decisions preserved:**
- 3-way missingness taxonomy: `observed_positive` (hit), `observed_null` (tested and not significant), `absent` (not tested, rare, or query failed).
- Network errors yield `absent` (`reason="query_failed"`), never `observed_null`.
- Structured JSON `evidence_value` avoiding CSV delimiter collisions.
- Full candidate $\times$ modality matrix completeness enforced via `validate_matrix_complete`.

In [ ]:
# Setup
!pip install -q requests pandas pyyaml

In [ ]:
import os
import json
import time
import hashlib
import subprocess
from collections import defaultdict, namedtuple
from dataclasses import dataclass, asdict
from datetime import datetime, timezone, timedelta
from typing import Optional, Callable

import requests
import pandas as pd

## 1. Configuration

In [ ]:
CONFIG_PATH = "configs/hbb.yaml"

DEFAULT_L1 = {
    "layer0_csv_path": None,            # optional explicit path to Layer 0's candidate CSV
    "strict": False,                    # if True, raises FileNotFoundError if Layer 0 CSV is missing
    "use_http_cache": True,             # caches network responses in data/cache/http to prevent API rate limits
    "http_cache_dir": "data/cache/http",
    "cache_ttl_days": 7,                # cache entries older than 7 days expire automatically
    "refresh_cache": False,             # set True to bypass cache and re-query live endpoints
    "gtex_dataset": "gtex_v8",
    "gtex_gencode_version": "v26",      # must match the dataset (gtex_v8 uses GENCODE v26)
    "gtex_tissues": ["Whole_Blood"],
    "gtex_items_per_page": 250,
    "gtex_max_pages": 40,
    "eqtl_testable_min_maf": 0.01,      # GTEx only tests variants with MAF >= 1% (gnomAD AF is a proxy)
    "gwas_p_threshold": 5.0e-8,
    "gwas_window_bp": 25000,            # cluster candidate positions to avoid querying massive locus spans
    "gnomad_dataset": "gnomad_r4",
    "gnomad_window_bp": 25000,
    "gnomad_request_delay_s": 6.0,      # public API is rate-limited (~10 req/min); only sleeps on real network calls
    "mpra_evidence_path": None,
    "crispr_evidence_path": None,
    "sig_pvalue_threshold": 0.05,       # nominal; supply FDR-adjusted p-values in your tables for real work
}

FALLBACK_CONFIG = {"gene_symbol": "HBB", "flank_bp": 100000, "layer1": {}}

if os.path.exists(CONFIG_PATH):
    import yaml
    with open(CONFIG_PATH) as f:
        CONFIG = yaml.safe_load(f)
else:
    CONFIG = FALLBACK_CONFIG
    print("No config file found -- using fallback CONFIG.")

GENE_SYMBOL = CONFIG["gene_symbol"]
FLANK_BP = CONFIG.get("flank_bp", 100000)
_user_l1 = CONFIG.get("layer1") or {}
L1 = {**DEFAULT_L1, **_user_l1}
_unknown = sorted(set(_user_l1) - set(DEFAULT_L1))
if _unknown:
    print(f"NOTE: ignoring unrecognised layer1 config keys: {_unknown}")
print(json.dumps(L1, indent=2))

## 2. Helpers — HTTP, Caching, Provenance

`HTTPCache` manages on-disk caching under `data/cache/http/`:
- **Cache Poisoning Prevention:** GraphQL responses with `"errors": [...]` and transient server errors are never cached.
- **404 Caching:** Persistent `http_404` errors are cached and returned directly as cache hits without redundant network calls.
- **TTL Support:** Entries older than `cache_ttl_days` automatically expire.
- **Cache Hit Transparency:** Returns `(payload, error, from_cache)` so rate-limiting sleeps are skipped on cache hits.

In [ ]:
class HTTPCache:
    def __init__(self, cache_dir: str, enabled: bool = True, ttl_days: int = 7, refresh: bool = False):
        self.cache_dir = cache_dir
        self.enabled = enabled
        self.ttl_days = ttl_days
        self.refresh = refresh
        self.hits = 0
        self.misses = 0
        self.oldest_entry_utc: Optional[str] = None
        if self.enabled and self.cache_dir:
            os.makedirs(self.cache_dir, exist_ok=True)

    def _cache_key(self, method: str, url: str, params, json_body) -> str:
        payload = {
            "m": method.upper(),
            "u": url,
            "p": sorted(params.items()) if params else None,
            "b": json_body
        }
        raw = json.dumps(payload, sort_keys=True, default=str).encode("utf-8")
        return hashlib.sha256(raw).hexdigest()

    def get(self, method: str, url: str, params, json_body):
        if not self.enabled or self.refresh or not self.cache_dir:
            return None, False
        ckey = self._cache_key(method, url, params, json_body)
        cpath = os.path.join(self.cache_dir, f"{ckey}.json")
        if not os.path.exists(cpath):
            return None, False
        try:
            with open(cpath, "r", encoding="utf-8") as f:
                entry = json.load(f)
            fetched_at_str = entry.get("fetched_at")
            if fetched_at_str:
                fetched_at = datetime.fromisoformat(fetched_at_str)
                now = datetime.now(timezone.utc)
                if now - fetched_at > timedelta(days=self.ttl_days):
                    return None, False  # expired TTL
                if self.oldest_entry_utc is None or fetched_at_str < self.oldest_entry_utc:
                    self.oldest_entry_utc = fetched_at_str
            self.hits += 1
            return entry, True
        except Exception:
            return None, False

    def put(self, method: str, url: str, params, json_body, data, error):
        if not self.enabled or not self.cache_dir:
            return
        # Do not cache transient failures, only clean data or persistent 404
        if error is not None and error != "http_404":
            return
        # Reject GraphQL responses with errors (cache poisoning prevention)
        if isinstance(data, dict) and data.get("errors"):
            return
        ckey = self._cache_key(method, url, params, json_body)
        cpath = os.path.join(self.cache_dir, f"{ckey}.json")
        now_str = datetime.now(timezone.utc).isoformat()
        try:
            with open(cpath, "w", encoding="utf-8") as f:
                json.dump({"data": data, "error": error, "fetched_at": now_str}, f)
            if self.oldest_entry_utc is None or now_str < self.oldest_entry_utc:
                self.oldest_entry_utc = now_str
        except Exception:
            pass


def make_http_fetch(cache: HTTPCache, timeout: int = 30, max_retries: int = 3, backoff: float = 2.0):
    def fetch(method: str, url: str, params=None, json_body=None):
        cached_entry, hit = cache.get(method, url, params, json_body)
        if hit:
            return cached_entry.get("data"), cached_entry.get("error"), True  # from_cache=True

        cache.misses += 1
        last_err = "unknown"
        for attempt in range(max_retries + 1):
            try:
                r = requests.request(method, url, params=params, json=json_body, timeout=timeout)
            except requests.exceptions.RequestException as e:
                last_err = type(e).__name__
            else:
                if r.status_code == 200:
                    try:
                        res = r.json()
                        cache.put(method, url, params, json_body, res, None)
                        return res, None, False
                    except ValueError:
                        return None, "invalid_json", False
                if r.status_code == 404:
                    cache.put(method, url, params, json_body, None, "http_404")
                    return None, "http_404", False
                if r.status_code in (429, 500, 502, 503, 504):
                    last_err = f"http_{r.status_code}"
                else:
                    return None, f"http_{r.status_code}", False
            if attempt < max_retries:
                time.sleep(backoff ** attempt)
        return None, last_err, False
    return fetch


def _file_hash(path, algo="sha256"):
    if not path or not os.path.exists(path):
        return None
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def _git_commit_hash():
    try:
        r = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True, timeout=5)
        return r.stdout.strip()
    except Exception:
        return None


def write_manifest(out_dir, filename, payload):
    manifest = {"generated_at_utc": datetime.now(timezone.utc).isoformat(), "git_commit": _git_commit_hash()}
    manifest.update(payload)
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, default=str)
    return path

## 3. Schema — `EvidenceRecord`, `MODALITY_META`, Validation

Long format: one row per (variant, modality, hit). `evidence_value` is compact JSON so downstream layers can parse it without delimiter ambiguity; non-positive rows carry a `reason`.

`MODALITY_META` specifies downstream handling across the framework roadmap:
- **Layer 2 (Rules & Completability):** Determines which `absent` modalities trigger computational models.
- **Layer 3 (AlphaGenome):** Completes modalities with `completable=True` (`eqtl`, `mpra`, `crispr`).
- **Layer 5 (Bayesian Evidence Fusion):** Fuses observations with `use_in_fusion=True` into a joint regulatory score.
- **Layer 6 (Calibration & Stratification):** Calibrates scores against ClinVar benchmarks; `gnomad_af` is used for population stratification, kept out of fusion to prevent label calibration leakage.

In [ ]:
SOURCE_TYPES = ("experimental", "computational", "curated_clinical")
MISSINGNESS_FLAGS = ("observed_positive", "observed_null", "absent")
MODALITIES = ("gwas", "eqtl", "gnomad_af", "mpra", "crispr")

MODALITY_META = {
    "gwas":      {"kind": "association",          "completable": False, "use_in_fusion": True,
                  "direction": "supports_regulatory_relevance"},
    "eqtl":      {"kind": "expression_effect",    "completable": True,  "use_in_fusion": True,
                  "direction": "supports_regulatory_relevance"},
    "mpra":      {"kind": "reporter_activity",    "completable": True,  "use_in_fusion": True,
                  "direction": "supports_regulatory_relevance"},
    "crispr":    {"kind": "perturbation_effect",  "completable": True,  "use_in_fusion": True,
                  "direction": "supports_regulatory_relevance"},
    "gnomad_af": {"kind": "population_frequency", "completable": False, "use_in_fusion": False,
                  "direction": "benign_leaning_when_observed"},
}


def _clean(o):
    """Make values JSON-safe: NaN/inf -> None, numpy scalars -> python, recurse into containers."""
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [_clean(v) for v in o]
    if hasattr(o, "item") and not isinstance(o, (str, bytes)):
        try:
            o = o.item()
        except Exception:
            pass
    if isinstance(o, float) and (o != o or o in (float("inf"), float("-inf"))):
        return None
    return o


def ev(**kw):
    return json.dumps(_clean(kw), sort_keys=True, allow_nan=False)


@dataclass
class EvidenceRecord:
    chrom: str
    pos: int
    ref: str
    alt: str
    modality: str
    source_type: str
    evidence_value: str
    context: Optional[str] = None
    missingness_flag: str = "observed_positive"

    def __post_init__(self):
        if self.modality not in MODALITIES:
            raise ValueError(f"modality must be one of {MODALITIES}, got {self.modality!r}")
        if self.source_type not in SOURCE_TYPES:
            raise ValueError(f"source_type must be one of {SOURCE_TYPES}, got {self.source_type!r}")
        if self.missingness_flag not in MISSINGNESS_FLAGS:
            raise ValueError(f"missingness_flag must be one of {MISSINGNESS_FLAGS}, got {self.missingness_flag!r}")


LAYER1_COLUMNS = ["chrom", "pos", "ref", "alt", "modality", "source_type",
                  "evidence_value", "context", "missingness_flag"]


def records_to_dataframe(records):
    if not records:
        return pd.DataFrame(columns=LAYER1_COLUMNS)
    return pd.DataFrame([asdict(r) for r in records])[LAYER1_COLUMNS]


def validate_layer1_output(df):
    missing = [c for c in LAYER1_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Layer 1 output missing required column(s): {missing}")
    for col, allowed in (("source_type", SOURCE_TYPES), ("missingness_flag", MISSINGNESS_FLAGS),
                         ("modality", MODALITIES)):
        bad = set(df[col].dropna()) - set(allowed)
        if bad:
            raise ValueError(f"Layer 1 output has invalid {col} value(s): {bad}")
    for s in df["evidence_value"]:
        json.loads(s)   # raises if any evidence_value is not valid JSON


def validate_matrix_complete(evidence_df, variants, modalities=MODALITIES):
    """Every candidate variant must have >= 1 record for every modality (downstream layers depend on this)."""
    expected = {v.key for v in variants}
    problems = []
    for m in modalities:
        sub = evidence_df[evidence_df["modality"] == m]
        got = {canon_key(r.chrom, r.pos, r.ref, r.alt) for r in sub[["chrom", "pos", "ref", "alt"]].itertuples(index=False)}
        missing = expected - got
        if missing:
            problems.append(f"{m}: {len(missing)} candidate variant(s) have no record")
    if problems:
        raise ValueError("Evidence matrix incomplete -- " + "; ".join(problems))

## 4. Helpers — Variant Identity & Genomic Windowing

- `canon_key` trims shared allele suffix/prefix so non-minimal indels match.
- `plan_windows` clusters variant positions into dense sub-windows (max span $\le W$) for region-based API queries (gnomAD and GWAS Catalog).

In [ ]:
Variant = namedtuple("Variant", "chrom pos ref alt key")


def canon_key(chrom, pos, ref, alt):
    ref, alt, pos = str(ref).upper(), str(alt).upper(), int(pos)
    while ref and alt and ref[-1] == alt[-1]:
        ref, alt = ref[:-1], alt[:-1]
    while ref and alt and ref[0] == alt[0]:
        ref, alt, pos = ref[1:], alt[1:], pos + 1
    return (str(chrom).replace("chr", ""), pos, ref, alt)


def plan_windows(positions, max_span):
    """Cluster sorted genomic positions into windows of at most max_span."""
    windows = []
    for p in sorted(set(positions)):
        if windows and p - windows[-1][0] < max_span:
            windows[-1][1] = p
        else:
            windows.append([p, p])
    return [(s, e) for s, e in windows]


def to_variants(df):
    out = []
    for r in df[["chrom", "pos", "ref", "alt"]].itertuples(index=False):
        chrom = str(r.chrom).replace("chr", "")
        out.append(Variant(chrom, int(r.pos), str(r.ref), str(r.alt), canon_key(chrom, r.pos, r.ref, r.alt)))
    return out


def make_record(v, modality, source_type, flag, context=None, **value):
    return EvidenceRecord(chrom=v.chrom, pos=v.pos, ref=v.ref, alt=v.alt, modality=modality,
                          source_type=source_type, evidence_value=ev(**value),
                          context=context, missingness_flag=flag)

## 5. Module Definitions

The following cells define the data loaders and API query collectors:
1. **Layer 0 Loader:** Reads candidates and gene metadata (pure functions).
2. **gnomAD:** Population allele frequency via GraphQL (region windows, skips delay on cache hits).
3. **GWAS Catalog:** Lead SNP associations via REST API (windowed queries with per-window error isolation).
4. **GTEx Portal:** Single-tissue eQTLs via REST API v2.
5. **Local Tables:** MPRA and CRISPR screen tables.

In [ ]:
def find_layer0_inputs(gene_symbol, explicit_csv=None):
    candidates = []
    if explicit_csv:
        candidates.append((explicit_csv, os.path.join(os.path.dirname(explicit_csv), "manifest.json"), "config path"))
    candidates.append((f"data/outputs/{gene_symbol}/layer0_regulatory_truthset.csv",
                       f"data/outputs/{gene_symbol}/manifest.json", "repo layout"))
    candidates.append((f"layer0_outputs/{gene_symbol}_layer0_regulatory_truthset.csv", None, "layer-0 notebook layout"))
    for csv_path, manifest_path, label in candidates:
        if os.path.exists(csv_path):
            return csv_path, manifest_path, label
    raise FileNotFoundError("No Layer 0 candidate CSV found. Looked in: " + ", ".join(c[0] for c in candidates))


def read_candidate_csv(csv_path):
    df = pd.read_csv(csv_path, dtype={"chrom": str})
    need = ["chrom", "pos", "ref", "alt"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} is missing required column(s): {missing}")
    df["chrom"] = df["chrom"].astype(str).str.replace("chr", "", regex=False)
    df["pos"] = df["pos"].astype(int)
    return df.drop_duplicates(subset=need).sort_values(["chrom", "pos"]).reset_index(drop=True)


def fetch_gene_metadata_fallback(gene_symbol, fetch):
    data, err, _ = fetch("GET", f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{gene_symbol}",
                         params={"content-type": "application/json"})
    if err:
        raise RuntimeError(f"Layer 0 manifest missing and Ensembl fallback lookup failed: {err}")
    return {"gene_symbol": gene_symbol, "gene_id": data["id"], "chrom": str(data["seq_region_name"]),
            "start": int(data["start"]), "end": int(data["end"]), "strand": int(data["strand"])}


def load_layer0_output(gene_symbol, explicit_csv=None, fetch=None):
    csv_path, manifest_path, layout = find_layer0_inputs(gene_symbol, explicit_csv)
    variants_df = read_candidate_csv(csv_path)
    info = {"csv_path": csv_path, "csv_sha256": _file_hash(csv_path), "layout": layout,
            "manifest_path": None, "n_variants": len(variants_df)}
    if manifest_path and os.path.exists(manifest_path):
        with open(manifest_path) as f:
            l0_manifest = json.load(f)
        gene_meta = l0_manifest["gene_metadata"]
        info["manifest_path"] = manifest_path
        info["clinvar_snapshot_date"] = l0_manifest.get("clinvar_snapshot_date")
    else:
        print("WARNING: no Layer 0 manifest found -- fetching gene metadata fresh from Ensembl. "
              "This is NOT pinned to the gene build/version Layer 0 used.")
        if fetch is None:
            raise ValueError("fetch function must be provided for fallback Ensembl gene metadata query.")
        gene_meta = fetch_gene_metadata_fallback(gene_symbol, fetch)
        info["gene_metadata_source"] = "fresh Ensembl lookup (not pinned to Layer 0)"
    return variants_df, gene_meta, info

### Module: gnomAD — population allele frequency (annotation only)

Region queries are planned from candidate positions clustered into windows (`gnomad_window_bp`).
Only `ac`/`an` are requested; AF is computed locally. Rate limit delays are applied only on actual network calls (cache hits return instantly).
gnomAD is `completable=False` (AlphaGenome cannot complete allele frequency) and `use_in_fusion=False` (prevents label calibration leakage).

In [ ]:
GNOMAD_API = "https://gnomad.broadinstitute.org/api"

GNOMAD_REGION_QUERY = """
query($chrom: String!, $start: Int!, $stop: Int!, $dataset: DatasetId!) {
  region(chrom: $chrom, start: $start, stop: $stop, reference_genome: GRCh38) {
    variants(dataset: $dataset) {
      variant_id
      pos
      ref
      alt
      genome { ac an }
    }
  }
}
"""


def collect_gnomad_evidence(variants, dataset, window_bp, delay_s, fetch):
    result = {"windows": [], "by_key": {}}
    by_chrom = defaultdict(list)
    for v in variants:
        by_chrom[v.chrom].append(v.pos)
    first_network_call = True
    for chrom, positions in by_chrom.items():
        for s, e in plan_windows(positions, window_bp):
            start, stop = max(1, s - 1), e + 1
            payload, err, from_cache = fetch("POST", GNOMAD_API, json_body={
                "query": GNOMAD_REGION_QUERY,
                "variables": {"chrom": chrom, "start": start, "stop": stop, "dataset": dataset}})
            # Delay only after actual network calls, never on cache hits
            if not from_cache and delay_s:
                if not first_network_call:
                    time.sleep(delay_s)
                first_network_call = False

            if not err and payload and payload.get("errors"):
                err = "graphql_error: " + str(payload["errors"][0].get("message", ""))[:120]
            win = {"chrom": chrom, "start": s, "stop": e, "error": err, "n_variants": 0, "from_cache": from_cache}
            if not err:
                region = ((payload or {}).get("data") or {}).get("region") or {}
                for gv in region.get("variants") or []:
                    genome = gv.get("genome")
                    if not genome or not genome.get("an") or not genome.get("ac"):
                        continue
                    key = canon_key(chrom, gv["pos"], gv["ref"], gv["alt"])
                    result["by_key"][key] = {"ac": genome["ac"], "an": genome["an"],
                                             "af": genome["ac"] / genome["an"],
                                             "variant_id": gv.get("variant_id")}
                    win["n_variants"] += 1
            result["windows"].append(win)
    return result


def build_gnomad_records(gnomad, variants, dataset):
    records = []
    for v in variants:
        hit = gnomad["by_key"].get(v.key)
        if hit:
            records.append(make_record(v, "gnomad_af", "curated_clinical", "observed_positive",
                                       context=f"{dataset}:genomes", af=hit["af"], ac=hit["ac"], an=hit["an"],
                                       gnomad_variant_id=hit["variant_id"]))
            continue
        failed = [w for w in gnomad["windows"] if w["chrom"] == v.chrom and w["start"] <= v.pos <= w["stop"] and w["error"]]
        if failed:
            records.append(make_record(v, "gnomad_af", "curated_clinical", "absent",
                                       context=f"{dataset}:genomes", reason="query_failed", error=failed[0]["error"]))
        else:
            records.append(make_record(v, "gnomad_af", "curated_clinical", "absent",
                                       context=f"{dataset}:genomes", reason="not_observed_in_genome_callset"))
    return records

### Module: GWAS — GWAS Catalog (Windowed with Per-Window Error Tracking)

Candidate positions are clustered using `plan_windows` (default 25kb max span, padded by $\pm 1$ bp) as a proactive precaution against querying wide intergenic intervals.
Errors are tracked per window: a network failure in one window marks only its enclosed variants as `incomplete_region_query`, while un-associated variants in successfully queried windows are correctly recorded as `not_in_catalog`.

In [ ]:
GWAS_BASE = "https://www.ebi.ac.uk/gwas/rest/api"


def _hal_embedded(payload, key):
    return ((payload or {}).get("_embedded") or {}).get(key) or []


def _clean_href(href):
    return href.split("{")[0] if href else href   # HAL templated links: ".../associations{?projection}"


def fetch_paginated_hal(url, params, embedded_key, fetch, max_pages=200):
    """Follow HAL _links.next. Returns (items, error); error != None means the item list is incomplete."""
    items, next_url, next_params, pages = [], url, dict(params or {}), 0
    while next_url and pages < max_pages:
        payload, err, _ = fetch("GET", next_url, params=next_params)
        if err:
            return items, err
        items.extend(_hal_embedded(payload, embedded_key))
        nxt = _clean_href(((payload or {}).get("_links") or {}).get("next", {}).get("href"))
        next_url, next_params = nxt, None
        pages += 1
    if next_url:
        return items, f"truncated_after_{max_pages}_pages"
    return items, None


def gwas_snp_position(snp, chrom):
    for loc in snp.get("locations") or []:
        if str(loc.get("chromosomeName")) == str(chrom) and loc.get("chromosomePosition") is not None:
            return int(loc["chromosomePosition"])
    return None


def assoc_pvalue(assoc):
    p = assoc.get("pvalue")
    if p is not None:
        try:
            return float(p)
        except (TypeError, ValueError):
            pass
    m, e = assoc.get("pvalueMantissa"), assoc.get("pvalueExponent")
    if m is not None and e is not None:
        try:
            return float(m) * 10 ** float(e)
        except (TypeError, ValueError):
            pass
    return None


def resolve_traits(assoc, fetch):
    inline = assoc.get("efoTraits")
    if inline:
        return [t.get("trait") for t in inline if t.get("trait")]
    emb = _hal_embedded(assoc, "efoTraits")
    if emb:
        return [t.get("trait") for t in emb if t.get("trait")]
    href = _clean_href((((assoc.get("_links") or {}).get("efoTraits")) or {}).get("href"))
    if href:
        payload, err, _ = fetch("GET", href)
        if not err:
            return [t.get("trait") for t in _hal_embedded(payload, "efoTraits") if t.get("trait")]
    return []


def collect_gwas_evidence(chrom, positions, fetch, size=500, window_bp=25000):
    result = {"windows": [], "by_pos": {}, "n_snps_in_region": 0}
    if not positions:
        return result
    wanted = set(positions)
    windows = plan_windows(positions, window_bp)
    all_snps = []
    for s, e in windows:
        # Pad bounds by +/- 1bp to guarantee inclusive matching
        start, stop = max(1, s - 1), e + 1
        snps, err = fetch_paginated_hal(
            f"{GWAS_BASE}/singleNucleotidePolymorphisms/search/findByChromBpLocationRange",
            {"chrom": str(chrom), "bpStart": start, "bpEnd": stop, "size": size},
            "singleNucleotidePolymorphisms", fetch)
        win = {"chrom": str(chrom), "start": s, "stop": e, "error": err, "n_snps": len(snps)}
        result["windows"].append(win)
        all_snps.extend(snps)

    result["n_snps_in_region"] = len(all_snps)
    for snp in all_snps:
        pos = gwas_snp_position(snp, chrom)
        if pos not in wanted:
            continue
        entry = result["by_pos"].setdefault(pos, {"rs_ids": [], "assocs": [], "error": None})
        rs = snp.get("rsId")
        if rs and rs not in entry["rs_ids"]:
            entry["rs_ids"].append(rs)
        links = snp.get("_links") or {}
        href = _clean_href(((links.get("snpAssociations") or links.get("associations")) or {}).get("href"))
        if not href:
            entry["error"] = entry["error"] or "no_association_link"
            continue
        assocs, aerr = fetch_paginated_hal(href, None, "associations", fetch)
        if aerr:
            entry["error"] = aerr
        for a in assocs:
            entry["assocs"].append({"traits": resolve_traits(a, fetch) or ["unresolved"],
                                    "pvalue": assoc_pvalue(a), "rs_id": rs})
    return result


def build_gwas_records(gwas, variants, p_threshold):
    records = []
    for v in variants:
        entry = gwas["by_pos"].get(v.pos)
        win = [w for w in gwas.get("windows", []) if w["chrom"] == str(v.chrom) and w["start"] <= v.pos <= w["stop"]]
        win_err = win[0]["error"] if win else None

        if entry is None:
            if win_err:
                records.append(make_record(v, "gwas", "experimental", "absent",
                                           reason="incomplete_region_query", error=win_err))
            else:
                records.append(make_record(v, "gwas", "experimental", "absent", reason="not_in_catalog"))
            continue
        ctx = (entry["rs_ids"] or [None])[0]
        sig = [a for a in entry["assocs"] if a["pvalue"] is not None and a["pvalue"] <= p_threshold]
        sub = [a for a in entry["assocs"] if a["pvalue"] is not None and a["pvalue"] > p_threshold]
        if sig:
            for a in sig:
                records.append(make_record(v, "gwas", "experimental", "observed_positive",
                                           context=a.get("rs_id") or ctx, traits=a["traits"], pvalue=a["pvalue"]))
        elif sub:
            records.append(make_record(v, "gwas", "experimental", "observed_null", context=ctx,
                                       reason="subthreshold_association",
                                       best_pvalue=min(a["pvalue"] for a in sub), n_assoc=len(sub)))
        else:
            records.append(make_record(v, "gwas", "experimental", "absent", context=ctx,
                                       reason="query_failed" if entry["error"] else "no_usable_association_records",
                                       error=entry["error"]))
    return records

### Module: eQTL — GTEx Portal API v2

GTEx only tests variants with MAF $\ge 1\%$ (gnomAD AF proxy).
A common variant with no hit is `observed_null`; a rare variant is `absent` (`not_testable_rare_or_unknown_frequency`).
GENCODE ID is resolved with version pinning to match the dataset.

In [ ]:
GTEX_BASE = "https://gtexportal.org/api/v2"


def resolve_versioned_gencode_id(gene_symbol, ensembl_id, gencode_version, fetch):
    attempts = ({"geneId": gene_symbol, "gencodeVersion": gencode_version, "genomeBuild": "GRCh38/hg38"},
                {"geneId": gene_symbol})
    for params in attempts:
        payload, err, _ = fetch("GET", f"{GTEX_BASE}/reference/gene", params=params)
        if err:
            continue
        records = (payload or {}).get("data") or []
        for rec in records:
            gid = rec.get("gencodeId") or ""
            if ensembl_id and gid.split(".")[0] == ensembl_id:
                return gid, None
        for rec in records:
            if rec.get("geneSymbol") == gene_symbol and rec.get("gencodeId"):
                return rec["gencodeId"], "matched_by_symbol_not_ensembl_id"
    return None, "gene_not_resolved"


def parse_gtex_variant_id(variant_id):
    if not variant_id:
        return None
    parts = variant_id.split("_")
    if len(parts) < 4:
        return None
    try:
        pos = int(parts[1])
    except ValueError:
        return None
    return parts[0].replace("chr", ""), pos, parts[2], parts[3]


def fetch_tissue_eqtls(gencode_id, tissue, dataset, fetch, items_per_page, max_pages):
    rows, seen = [], set()
    for page in range(max_pages):
        payload, err, _ = fetch("GET", f"{GTEX_BASE}/association/singleTissueEqtl",
                                params={"gencodeId": gencode_id, "tissueSiteDetailId": tissue,
                                        "datasetId": dataset, "itemsPerPage": items_per_page, "page": page})
        if err:
            return rows, err
        data = (payload or {}).get("data") or []
        new = 0
        for rec in data:
            vid = rec.get("variantId")
            if vid in seen:
                continue
            seen.add(vid)
            rows.append(rec)
            new += 1
        n_pages = ((payload or {}).get("paging_info") or {}).get("numberOfPages")
        if len(data) < items_per_page:
            return rows, None
        if new == 0:
            return rows, "pagination_not_advancing"
        if n_pages is not None and page + 1 >= n_pages:
            return rows, None
    return rows, f"truncated_after_{max_pages}_pages"


def collect_eqtl_evidence(gene_meta, tissues, dataset, gencode_version, items_per_page, max_pages, fetch):
    out = {"gencode_id": None, "resolve_note": None, "resolve_error": None, "tissues": {}}
    gid, note = resolve_versioned_gencode_id(gene_meta["gene_symbol"], gene_meta.get("gene_id"), gencode_version, fetch)
    if gid is None:
        out["resolve_error"] = note
        return out
    out["gencode_id"], out["resolve_note"] = gid, note
    for tissue in tissues:
        rows, err = fetch_tissue_eqtls(gid, tissue, dataset, fetch, items_per_page, max_pages)
        by_key = defaultdict(list)
        for rec in rows:
            parsed = parse_gtex_variant_id(rec.get("variantId"))
            if not parsed:
                continue
            by_key[canon_key(*parsed)].append({"slope": rec.get("nes", rec.get("slope")),
                                                "pvalue": rec.get("pValue", rec.get("pval")),
                                                "variant_id": rec.get("variantId")})
        out["tissues"][tissue] = {"error": err, "n_rows": len(rows), "by_key": dict(by_key)}
    return out


def build_eqtl_records(eqtl, variants, tissues, maf_by_key, min_maf):
    records = []
    for tissue in tissues:
        tinfo = eqtl["tissues"].get(tissue)
        for v in variants:
            if eqtl.get("resolve_error") or tinfo is None:
                records.append(make_record(v, "eqtl", "experimental", "absent", context=tissue,
                                           reason="query_failed",
                                           error=eqtl.get("resolve_error") or "tissue_not_queried"))
                continue
            hits = tinfo["by_key"].get(v.key)
            if hits:
                for h in hits:
                    records.append(make_record(v, "eqtl", "experimental", "observed_positive", context=tissue,
                                               slope=h["slope"], pvalue=h["pvalue"], gtex_variant_id=h["variant_id"]))
                continue
            if tinfo["error"]:
                records.append(make_record(v, "eqtl", "experimental", "absent", context=tissue,
                                           reason="incomplete_tissue_query", error=tinfo["error"]))
                continue
            maf = maf_by_key.get(v.key)
            if maf is not None and maf >= min_maf:
                records.append(make_record(v, "eqtl", "experimental", "observed_null", context=tissue,
                                           reason="common_variant_not_significant", maf=maf))
            else:
                records.append(make_record(v, "eqtl", "experimental", "absent", context=tissue,
                                           reason="not_testable_rare_or_unknown_frequency", maf=maf))
    return records

### Modules: MPRA and CRISPR — Local Evidence Tables

Reads optional tabular files (CSV/TSV):
- in table, $p \le p_{thresh}$ -> `observed_positive`
- in table, $p > p_{thresh}$ -> `observed_null` (tested, not significant)
- not in table -> `absent` (`not_tested_in_provided_table`)
- no path configured -> `absent` (`no_data_source_configured`)

In [ ]:
MPRA_COLUMNS = ["chrom", "pos", "ref", "alt", "cell_type", "log2_allelic_skew", "pvalue", "dataset_name"]
CRISPR_COLUMNS = ["chrom", "pos", "ref", "alt", "target_gene", "cell_type", "effect_size", "pvalue",
                  "screen_type", "dataset_name"]


def load_local_evidence_table(path, expected_columns, label, numeric_columns=()):
    if not path:
        return None
    if not os.path.exists(path):
        raise FileNotFoundError(f"{label} evidence path is configured but the file does not exist: {path}")
    df = pd.read_csv(path, sep=None, engine="python", dtype=str)
    missing = [c for c in expected_columns if c not in df.columns]
    if missing:
        raise ValueError(f"{label} evidence table at {path} is missing expected column(s): {missing}. "
                         f"Expected: {expected_columns}")
    df["chrom"] = df["chrom"].astype(str).str.replace("chr", "", regex=False)
    df["pos"] = pd.to_numeric(df["pos"], errors="coerce")
    n_bad = int(df["pos"].isna().sum())
    if n_bad:
        print(f"WARNING: {label} table has {n_bad} row(s) with a non-numeric pos; dropped.")
        df = df.dropna(subset=["pos"])
    df["pos"] = df["pos"].astype(int)
    df["ref"], df["alt"] = df["ref"].str.upper(), df["alt"].str.upper()
    for c in ["pvalue", *numeric_columns]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    index = defaultdict(list)
    for row in df.to_dict("records"):
        index[canon_key(row["chrom"], row["pos"], row["ref"], row["alt"])].append(row)
    return dict(index)


def build_local_table_records(modality, variants, index, p_threshold, value_fn):
    records = []
    for v in variants:
        if index is None:
            records.append(make_record(v, modality, "experimental", "absent", reason="no_data_source_configured"))
            continue
        rows = index.get(v.key)
        if not rows:
            records.append(make_record(v, modality, "experimental", "absent", reason="not_tested_in_provided_table"))
            continue
        for row in rows:
            p, ctx = row.get("pvalue"), row.get("cell_type")
            if p is None or p != p:
                records.append(make_record(v, modality, "experimental", "absent", context=ctx, reason="p_value_missing"))
            elif p <= p_threshold:
                records.append(make_record(v, modality, "experimental", "observed_positive", context=ctx,
                                           **value_fn(row)))
            else:
                records.append(make_record(v, modality, "experimental", "observed_null", context=ctx,
                                           reason="tested_not_significant", pvalue=p))
    return records


def mpra_value(row):
    return {"log2_allelic_skew": row.get("log2_allelic_skew"), "pvalue": row.get("pvalue"),
            "dataset": row.get("dataset_name")}


def crispr_value(row):
    return {"target_gene": row.get("target_gene"), "effect_size": row.get("effect_size"),
            "pvalue": row.get("pvalue"), "screen_type": row.get("screen_type"),
            "dataset": row.get("dataset_name")}

## 6. Intake — Load Layer 0 Candidate Truthset

In [ ]:
# Initialize cache and bound fetch function
http_cache = HTTPCache(
    cache_dir=L1["http_cache_dir"],
    enabled=L1["use_http_cache"],
    ttl_days=L1.get("cache_ttl_days", 7),
    refresh=L1.get("refresh_cache", False)
)
fetch = make_http_fetch(http_cache)

try:
    variants_df, gene_meta, layer0_info = load_layer0_output(GENE_SYMBOL, L1["layer0_csv_path"], fetch=fetch)
    variants = to_variants(variants_df)
    print(f"Loaded {len(variants)} candidate variants for {GENE_SYMBOL} from {layer0_info['csv_path']} ({layer0_info['layout']})")
    print(f"Gene metadata: {gene_meta}")
except FileNotFoundError as e:
    if L1.get("strict"):
        raise
    variants_df, gene_meta, layer0_info, variants = None, None, None, None
    print(f"INFO: Layer 0 outputs not found yet ({e}).")
    print("To run live collection, execute Layer 0 first to produce candidate CSVs.")
    print("You can still proceed to Section 10 to run the full self-contained test suite.")

## 7. Run Evidence Collection & Matrix Assembly

In [ ]:
if variants is None:
    print("Skipping live collection: Layer 0 candidate variants not loaded.")
    evidence_df = None
else:
    RUN_WARNINGS = []
    if {v.chrom for v in variants} != {str(gene_meta["chrom"])}:
        raise ValueError("Candidate variants span more than the gene's chromosome; "
                         "multi-chromosome candidate sets are not supported in this notebook.")
    chrom = str(gene_meta["chrom"])

    # --- gnomAD ---
    print("Collecting gnomAD allele frequencies...")
    gnomad = collect_gnomad_evidence(variants, L1["gnomad_dataset"], L1["gnomad_window_bp"],
                                     L1["gnomad_request_delay_s"], fetch=fetch)
    maf_by_key = {k: min(h["af"], 1 - h["af"]) for k, h in gnomad["by_key"].items()}
    for w in gnomad["windows"]:
        if w["error"]:
            RUN_WARNINGS.append(f"gnomAD window {w['chrom']}:{w['start']}-{w['stop']} failed: {w['error']}")
    print(f"gnomAD: {len(gnomad['windows'])} window(s), {len(gnomad['by_key'])} variant(s) with genome AF")

    # --- GWAS Catalog (Windowed) ---
    print("Collecting GWAS Catalog associations (windowed)...")
    positions = sorted({v.pos for v in variants})
    gwas = collect_gwas_evidence(chrom, positions, fetch=fetch, window_bp=L1.get("gwas_window_bp", 25000))
    failed_gwas_wins = [w for w in gwas["windows"] if w["error"]]
    if failed_gwas_wins:
        RUN_WARNINGS.append(f"GWAS: {len(failed_gwas_wins)} of {len(gwas['windows'])} window(s) failed query")
    print(f"GWAS: {gwas['n_snps_in_region']} catalog SNP(s) across {len(gwas['windows'])} window(s), "
          f"{len(gwas['by_pos'])} at candidate positions")

    # --- GTEx eQTL ---
    print("Collecting GTEx eQTLs...")
    tissues = list(L1["gtex_tissues"])
    eqtl = collect_eqtl_evidence(gene_meta, tissues, L1["gtex_dataset"], L1["gtex_gencode_version"],
                                 L1["gtex_items_per_page"], L1["gtex_max_pages"], fetch=fetch)
    if eqtl["resolve_error"]:
        RUN_WARNINGS.append(f"GTEx gene resolution failed: {eqtl['resolve_error']}")
    if eqtl["resolve_note"]:
        RUN_WARNINGS.append(f"GTEx gene resolution note: {eqtl['resolve_note']}")
    for t, info in eqtl["tissues"].items():
        if info["error"]:
            RUN_WARNINGS.append(f"GTEx tissue {t}: {info['error']} after {info['n_rows']} row(s)")
        elif info["n_rows"] == 0:
            RUN_WARNINGS.append(f"GTEx tissue {t}: 0 significant eQTL rows returned for {gene_meta['gene_symbol']}")
    print(f"GTEx: gencode_id={eqtl['gencode_id']}, " +
          ", ".join(f"{t}={i['n_rows']} rows" for t, i in eqtl["tissues"].items()))

    # --- Local Tables (MPRA & CRISPR) ---
    mpra_index = load_local_evidence_table(L1["mpra_evidence_path"], MPRA_COLUMNS, "MPRA", ("log2_allelic_skew",))
    crispr_index = load_local_evidence_table(L1["crispr_evidence_path"], CRISPR_COLUMNS, "CRISPR", ("effect_size",))

    # --- Assemble All Records ---
    sig_p = L1["sig_pvalue_threshold"]
    all_records = (
        build_gwas_records(gwas, variants, L1["gwas_p_threshold"])
        + build_eqtl_records(eqtl, variants, tissues, maf_by_key, L1["eqtl_testable_min_maf"])
        + build_gnomad_records(gnomad, variants, L1["gnomad_dataset"])
        + build_local_table_records("mpra", variants, mpra_index, sig_p, mpra_value)
        + build_local_table_records("crispr", variants, crispr_index, sig_p, crispr_value)
    )
    evidence_df = records_to_dataframe(all_records)
    validate_layer1_output(evidence_df)
    validate_matrix_complete(evidence_df, variants)
    print(f"Total evidence records: {len(evidence_df)} (complete matrix for all {len(variants)} candidates x {len(MODALITIES)} modalities)")

## 8. Save Output + Provenance Manifest

In [ ]:
if evidence_df is not None:
    out_dir = os.path.join("data/outputs", GENE_SYMBOL)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, "layer1_evidence.csv")
    evidence_df.to_csv(out_path, index=False)

    flag_counts = {m: {f: int(n) for f, n in g["missingness_flag"].value_counts().items()}
                   for m, g in evidence_df.groupby("modality")}

    manifest_path = write_manifest(out_dir, "layer1_manifest.json", {
        "layer": 1,
        "notebook_version": "layer1_v5",
        "gene_symbol": GENE_SYMBOL,
        "gene_metadata": gene_meta,
        "layer0_input": layer0_info,
        "config_path": CONFIG_PATH,
        "config_sha256": _file_hash(CONFIG_PATH),
        "layer1_effective_config": L1,
        "modality_meta": MODALITY_META,
        "n_candidate_variants": len(variants),
        "n_evidence_records": len(evidence_df),
        "flag_counts": flag_counts,
        "cache": {
            "enabled": L1["use_http_cache"],
            "cache_dir": L1["http_cache_dir"],
            "hits": http_cache.hits,
            "misses": http_cache.misses,
            "oldest_entry_utc": http_cache.oldest_entry_utc
        },
        "gtex": {"gencode_id": eqtl["gencode_id"], "dataset": L1["gtex_dataset"],
                 "tissues": {t: {"n_rows": i["n_rows"], "error": i["error"]} for t, i in eqtl["tissues"].items()}},
        "gnomad": {"dataset": L1["gnomad_dataset"], "windows": gnomad["windows"]},
        "gwas": {"windows": gwas["windows"], "n_snps_in_span": gwas["n_snps_in_region"]},
        "mpra_evidence_path": L1["mpra_evidence_path"], "mpra_sha256": _file_hash(L1["mpra_evidence_path"]),
        "crispr_evidence_path": L1["crispr_evidence_path"], "crispr_sha256": _file_hash(L1["crispr_evidence_path"]),
        "any_query_failed": bool(RUN_WARNINGS),
        "warnings": RUN_WARNINGS,
    })
    print(f"Saved {len(evidence_df)} records to {out_path}")
    print(f"Manifest: {manifest_path}")
    print(f"Cache performance: {http_cache.hits} hits, {http_cache.misses} misses (oldest entry: {http_cache.oldest_entry_utc})")
    if RUN_WARNINGS:
        print(f"\n{len(RUN_WARNINGS)} WARNING(S) -- affected variants are marked absent, not null:")
        for w in RUN_WARNINGS:
            print("  -", w)
else:
    print("No evidence DataFrame to save.")

## 9. Inspect Results

In [ ]:
if evidence_df is not None:
    display(evidence_df.groupby(["modality", "missingness_flag"]).size().unstack(fill_value=0))

In [ ]:
if evidence_df is not None:
    reasons = evidence_df.assign(reason=evidence_df["evidence_value"].map(lambda s: json.loads(s).get("reason", "(hit)")))
    display(reasons.groupby(["modality", "missingness_flag", "reason"]).size())

## 10. Regression / Sanity Check Suite

Self-contained unit and regression test suite with deterministic fake responses.
Runs completely offline without external network dependencies.

In [ ]:
import tempfile


def _fake_fetch(routes):
    calls = []
    def fetch(method, url, params=None, json_body=None):
        calls.append((method, url, dict(params) if params else None))
        for match, respond in routes:
            if match(method, url, params, json_body):
                data, err = respond(method, url, params, json_body)
                return data, err, False
        return None, "http_404", False
    fetch.calls = calls
    return fetch


def _variants(rows):
    return to_variants(pd.DataFrame([{"chrom": "11", "pos": p, "ref": r, "alt": a} for p, r, a in rows]))


def _flag(recs, pos, context=None):
    return [r for r in recs if r.pos == pos and (context is None or r.context == context)]


def test_canon_key_and_gtex_ids():
    assert canon_key("chr11", 100, "A", "G") == ("11", 100, "A", "G")
    assert canon_key("11", 5227100, "G", "GA") == canon_key("11", 5227099, "AG", "AGA")
    assert parse_gtex_variant_id("chr11_5227002_T_C_b38") == ("11", 5227002, "T", "C")
    assert parse_gtex_variant_id("chr1_55516888_G_GA_b38") == ("1", 55516888, "G", "GA")
    assert parse_gtex_variant_id("nope") is None and parse_gtex_variant_id(None) is None
    print("PASS: canonical keys and GTEx variant-id parsing")


def test_http_cache_and_poisoning_prevention():
    with tempfile.TemporaryDirectory() as d:
        cache = HTTPCache(cache_dir=d, enabled=True, ttl_days=7)
        # 1. Successful response cached
        cache.put("GET", "https://api.test/data", {"a": "1"}, None, {"test": 123}, None)
        entry, hit = cache.get("GET", "https://api.test/data", {"a": "1"}, None)
        assert hit and entry.get("data") == {"test": 123}

        # 2. GraphQL response with errors must NOT be cached (poisoning prevention)
        cache.put("POST", "https://api.test/gql", None, {"q": 1}, {"errors": [{"msg": "fail"}]}, None)
        _, hit = cache.get("POST", "https://api.test/gql", None, {"q": 1})
        assert not hit, "GraphQL response with errors must not be cached"

        # 3. 404 response cached and retrieved
        cache.put("GET", "https://api.test/404", None, None, None, "http_404")
        entry, hit = cache.get("GET", "https://api.test/404", None, None)
        assert hit and entry.get("error") == "http_404"

        # 4. TTL expiration
        old_time = (datetime.now(timezone.utc) - timedelta(days=8)).isoformat()
        cache.put("GET", "https://api.test/old", None, None, {"old": True}, None)
        ckey = cache._cache_key("GET", "https://api.test/old", None, None)
        with open(os.path.join(d, f"{ckey}.json"), "w") as f:
            json.dump({"data": {"old": True}, "error": None, "fetched_at": old_time}, f)
        _, hit = cache.get("GET", "https://api.test/old", None, None)
        assert not hit, "Expired TTL must yield cache miss"
    print("PASS: HTTP cache storage, 404 caching, TTL expiration, and poisoning prevention")


def test_cache_config_toggle():
    with tempfile.TemporaryDirectory() as d:
        cache_off = HTTPCache(cache_dir=d, enabled=False)
        cache_off.put("GET", "https://api.test/x", None, None, {"x": 1}, None)
        assert len(os.listdir(d)) == 0, "Disabled cache must write 0 files"
    print("PASS: Cache config toggle respects enabled=False")


def test_gwas_multi_window_failure_isolation():
    variants = _variants([(100, "A", "G"), (50000, "C", "T"), (100000, "G", "A")])
    snp = lambda rs, pos: {"rsId": rs, "locations": [{"chromosomeName": "11", "chromosomePosition": pos}],
                           "_links": {"snpAssociations": {"href": f"https://x/assoc/{rs}"}}}

    def fake_fetch(method, url, params=None, json_body=None):
        if "findByChromBpLocationRange" in url:
            start = params["bpStart"]
            if start < 1000:       # Window 1: fails
                return None, "http_500", False
            elif start < 60000:    # Window 2: healthy, empty
                return {"_embedded": {"singleNucleotidePolymorphisms": []}}, None, False
            else:                  # Window 3: healthy, has hit
                return {"_embedded": {"singleNucleotidePolymorphisms": [snp("rs3", 100000)]}}, None, False
        if "/assoc/" in url:
            return {"_embedded": {"associations": [{"pvalue": 1e-9, "efoTraits": [{"trait": "HbF"}]}]}}, None, False
        return None, "http_404", False

    gwas = collect_gwas_evidence("11", [100, 50000, 100000], fetch=fake_fetch, window_bp=25000)
    recs = build_gwas_records(gwas, variants, 5e-8)
    r100 = [r for r in recs if r.pos == 100][0]
    r50k = [r for r in recs if r.pos == 50000][0]
    r100k = [r for r in recs if r.pos == 100000][0]

    assert r100.missingness_flag == "absent" and json.loads(r100.evidence_value)["reason"] == "incomplete_region_query"
    assert r50k.missingness_flag == "absent" and json.loads(r50k.evidence_value)["reason"] == "not_in_catalog"
    assert r100k.missingness_flag == "observed_positive"
    print("PASS: Multi-window GWAS failure isolated to failed window only")


def test_gwas_pagination_scope_and_semantics():
    snp = lambda rs, pos: {"rsId": rs, "locations": [{"chromosomeName": "11", "chromosomePosition": pos}],
                           "_links": {"snpAssociations": {"href": f"https://x/assoc/{rs}{{?projection}}"}}}

    def page1(m, u, p, b):
        return {"_embedded": {"singleNucleotidePolymorphisms": [snp("rs1", 100), snp("rs2", 999)]},
                "_links": {"next": {"href": "https://x/region?page=1"}}}, None

    def page2(m, u, p, b):
        return {"_embedded": {"singleNucleotidePolymorphisms": [snp("rs3", 300)]}, "_links": {}}, None

    def assoc(m, u, p, b):
        pv = {"rs1": 1e-9, "rs3": 1e-6}[u.split("/assoc/")[1]]
        return {"_embedded": {"associations": [{"pvalue": pv, "efoTraits": [{"trait": "HbF, level"}]}]}}, None

    fetch = _fake_fetch([(lambda m, u, p, b: "findByChromBpLocationRange" in u, page1),
                         (lambda m, u, p, b: "region?page=1" in u, page2),
                         (lambda m, u, p, b: "/assoc/" in u, assoc)])
    res = collect_gwas_evidence("11", [100, 300, 500], fetch=fetch, window_bp=25000)
    assert set(res["by_pos"]) == {100, 300}, "SNP on the second page must be found"
    assoc_calls = [c for c in fetch.calls if "/assoc/" in c[1]]
    assert len(assoc_calls) == 2 and not any("{" in c[1] for c in fetch.calls), \
        "association follow-up only for candidate positions, with templated hrefs cleaned"

    recs = build_gwas_records(res, _variants([(100, "A", "G"), (300, "C", "T"), (500, "G", "A")]), 5e-8)
    r100, r300, r500 = _flag(recs, 100)[0], _flag(recs, 300)[0], _flag(recs, 500)[0]
    assert r100.missingness_flag == "observed_positive" and json.loads(r100.evidence_value)["traits"] == ["HbF, level"]
    assert r300.missingness_flag == "observed_null" and json.loads(r300.evidence_value)["reason"] == "subthreshold_association"
    assert r500.missingness_flag == "absent" and json.loads(r500.evidence_value)["reason"] == "not_in_catalog"
    print("PASS: GWAS pagination, windowing scope, positive / sub-threshold null / not-in-catalog absent")


def test_eqtl_testability_and_failed_tissue():
    variants = _variants([(100, "A", "G"), (200, "C", "T"), (300, "G", "A"), (400, "T", "C")])
    eqtl = {"gencode_id": "ENSG1.1", "resolve_note": None, "resolve_error": None, "tissues": {
        "Whole_Blood": {"error": None, "n_rows": 1, "by_key": {
            canon_key("11", 100, "A", "G"): [{"slope": 0.4, "pvalue": 1e-8, "variant_id": "chr11_100_A_G_b38"}]}},
        "Liver": {"error": "http_500", "n_rows": 0, "by_key": {}}}}
    maf = {canon_key("11", 200, "C", "T"): 0.2, canon_key("11", 300, "G", "A"): 0.0001}
    recs = build_eqtl_records(eqtl, variants, ["Whole_Blood", "Liver"], maf, 0.01)
    assert _flag(recs, 100, "Whole_Blood")[0].missingness_flag == "observed_positive"
    assert _flag(recs, 200, "Whole_Blood")[0].missingness_flag == "observed_null"
    assert _flag(recs, 300, "Whole_Blood")[0].missingness_flag == "absent"
    assert json.loads(_flag(recs, 300, "Whole_Blood")[0].evidence_value)["reason"].startswith("not_testable")
    assert _flag(recs, 400, "Whole_Blood")[0].missingness_flag == "absent"
    assert all(r.missingness_flag == "absent" for r in recs if r.context == "Liver")
    print("PASS: eQTL rare/unknown -> absent, common non-hit -> null, failed tissue -> absent")


def test_eqtl_pagination_truncation_and_stuck_pages():
    rows = [{"variantId": f"chr11_{i}_A_G_b38", "nes": 0.1, "pValue": 1e-6} for i in range(5)]

    def paged(m, u, p, b):
        chunk = rows[p["page"] * p["itemsPerPage"]:(p["page"] + 1) * p["itemsPerPage"]]
        return {"data": chunk, "paging_info": {"numberOfPages": 3, "page": p["page"]}}, None

    route = [(lambda m, u, p, b: "singleTissueEqtl" in u, paged)]
    got, err = fetch_tissue_eqtls("E.1", "Whole_Blood", "gtex_v8", _fake_fetch(route), 2, 10)
    assert err is None and len(got) == 5, "all pages must be collected"
    got, err = fetch_tissue_eqtls("E.1", "Whole_Blood", "gtex_v8", _fake_fetch(route), 2, 2)
    assert err and err.startswith("truncated"), "hitting the page cap must be reported"
    stuck = [(lambda m, u, p, b: True, lambda m, u, p, b: ({"data": rows[:2]}, None))]
    got, err = fetch_tissue_eqtls("E.1", "Whole_Blood", "gtex_v8", _fake_fetch(stuck), 2, 10)
    assert err == "pagination_not_advancing"
    print("PASS: GTEx pagination, truncation flag, non-advancing-page flag")


def test_gnomad_windows_af_and_delay_skipping():
    variants = _variants([(100, "A", "G"), (200, "C", "T"), (900000, "G", "A")])
    assert plan_windows([v.pos for v in variants], 25000) == [(100, 200), (900000, 900000)]

    def fake_cached_fetch(method, url, params=None, json_body=None):
        if json_body and json_body.get("variables", {}).get("start", 0) > 800000:
            return None, "http_503", False
        return {"data": {"region": {"variants": [
            {"variant_id": "11-100-A-G", "pos": 100, "ref": "A", "alt": "G", "genome": {"ac": 3, "an": 150000}},
            {"variant_id": "11-150-A-G", "pos": 150, "ref": "A", "alt": "G", "genome": None}]}}}, None, True  # from_cache=True

    t0 = time.time()
    res = collect_gnomad_evidence(variants, "gnomad_r4", 25000, delay_s=5.0, fetch=fake_cached_fetch)
    elapsed = time.time() - t0
    assert elapsed < 0.5, "gnomAD cache hits must skip rate-limit sleep"

    recs = {r.pos: r for r in build_gnomad_records(res, variants, "gnomad_r4")}
    assert recs[100].missingness_flag == "observed_positive"
    assert abs(json.loads(recs[100].evidence_value)["af"] - 3 / 150000) < 1e-12
    assert json.loads(recs[200].evidence_value)["reason"] == "not_observed_in_genome_callset"
    assert json.loads(recs[900000].evidence_value)["reason"] == "query_failed"
    print("PASS: gnomAD windows, AF calculation, failed window isolation, and cache delay skip")


def test_local_tables():
    variants = _variants([(100, "A", "G"), (200, "C", "T"), (300, "G", "A"), (400, "T", "C")])
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, "mpra.csv")
        pd.DataFrame([
            {"chrom": "11", "pos": 100, "ref": "A", "alt": "G", "cell_type": "K562", "log2_allelic_skew": 1.8, "pvalue": 0.001, "dataset_name": "t"},
            {"chrom": "11", "pos": 200, "ref": "C", "alt": "T", "cell_type": "K562", "log2_allelic_skew": 0.1, "pvalue": 0.4, "dataset_name": "t"},
            {"chrom": "11", "pos": 300, "ref": "G", "alt": "A", "cell_type": "K562", "log2_allelic_skew": 0.1, "pvalue": None, "dataset_name": "t"},
        ]).to_csv(path, index=False)
        idx = load_local_evidence_table(path, MPRA_COLUMNS, "MPRA", ("log2_allelic_skew",))
        recs = build_local_table_records("mpra", variants, idx, 0.05, mpra_value)
        by = {r.pos: r for r in recs}
        assert by[100].missingness_flag == "observed_positive"
        assert by[200].missingness_flag == "observed_null"
        assert by[300].missingness_flag == "absent" and json.loads(by[300].evidence_value)["reason"] == "p_value_missing"
        assert by[400].missingness_flag == "absent" and json.loads(by[400].evidence_value)["reason"] == "not_tested_in_provided_table"

        bad = os.path.join(d, "bad.csv")
        pd.DataFrame([{"chrom": "11", "pos": 1}]).to_csv(bad, index=False)
        for fn, exc in ((lambda: load_local_evidence_table(bad, MPRA_COLUMNS, "MPRA"), ValueError),
                        (lambda: load_local_evidence_table(os.path.join(d, "nope.csv"), MPRA_COLUMNS, "MPRA"), FileNotFoundError)):
            try:
                fn()
                raise AssertionError("expected " + exc.__name__)
            except exc:
                pass
    none_recs = build_local_table_records("crispr", variants, None, 0.05, crispr_value)
    assert all(r.missingness_flag == "absent" and json.loads(r.evidence_value)["reason"] == "no_data_source_configured" for r in none_recs)
    print("PASS: local tables -- positive / tested-null / missing-p / not-in-table absent / schema enforcement")


def test_validators_and_json():
    variants = _variants([(100, "A", "G"), (200, "C", "T")])
    full = build_local_table_records("mpra", variants, None, 0.05, mpra_value)
    df = records_to_dataframe(full)
    try:
        validate_matrix_complete(df.iloc[:1], variants, modalities=("mpra",))
        raise AssertionError("expected incomplete-matrix error")
    except ValueError:
        pass
    validate_matrix_complete(df, variants, modalities=("mpra",))
    assert json.loads(ev(x=float("nan"), y=[1, float("inf")])) == {"x": None, "y": [1, None]}
    bad = df.copy()
    bad.loc[0, "source_type"] = "made_up"
    try:
        validate_layer1_output(bad)
        raise AssertionError("expected invalid source_type error")
    except ValueError:
        pass
    print("PASS: matrix-completeness validator, NaN-safe JSON, schema validation")


def test_layer0_discovery_and_strict_mode():
    old = os.getcwd()
    with tempfile.TemporaryDirectory() as d:
        os.chdir(d)
        try:
            os.makedirs("layer0_outputs")
            pd.DataFrame([{"chrom": 11, "pos": 5, "ref": "A", "alt": "G"}]).to_csv(
                "layer0_outputs/HBB_layer0_regulatory_truthset.csv", index=False)
            _, manifest, label = find_layer0_inputs("HBB")
            assert label == "layer-0 notebook layout" and manifest is None
            os.makedirs("data/outputs/HBB")
            pd.DataFrame([{"chrom": 11, "pos": 5, "ref": "A", "alt": "G"}]).to_csv(
                "data/outputs/HBB/layer0_regulatory_truthset.csv", index=False)
            path, _, label = find_layer0_inputs("HBB")
            assert label == "repo layout"
            df = read_candidate_csv(path)
            assert df["chrom"].iloc[0] == "11" and isinstance(df["chrom"].iloc[0], str)
            try:
                find_layer0_inputs("MISSING_GENE")
                raise AssertionError("expected FileNotFoundError")
            except FileNotFoundError:
                pass
        finally:
            os.chdir(old)
    print("PASS: Layer 0 discovery and string chrom dtypes")


test_canon_key_and_gtex_ids()
test_http_cache_and_poisoning_prevention()
test_cache_config_toggle()
test_gwas_multi_window_failure_isolation()
test_gwas_pagination_scope_and_semantics()
test_eqtl_testability_and_failed_tissue()
test_eqtl_pagination_truncation_and_stuck_pages()
test_gnomad_windows_af_and_delay_skipping()
test_local_tables()
test_validators_and_json()
test_layer0_discovery_and_strict_mode()
print("\nALL 11 REGRESSION AND SANITY CHECKS PASSED!")

## 11. Handoff to Layer 2 & Layer 3 (AlphaGenome)

`evidence_df` gives downstream layers a complete (variant × modality) matrix. Handoff rules across the roadmap:

- **Layer 2 (Rules & Completability):** Evaluates `absent` flags. Triggers Layer 3 (AlphaGenome) only for `(variant, modality)` pairs that are `absent` **and** `MODALITY_META[modality]["completable"]` is true (`eqtl`, `mpra`, `crispr`). Read `MODALITY_META` from `layer1_manifest.json`.
- **Reason Interpretation:** The `reason` field inside `evidence_value` distinguishes biological gaps from technical failures:
  - `not_testable_rare_or_unknown_frequency` and `not_tested_in_provided_table` are true evidence gaps requiring AlphaGenome completion.
  - `query_failed` and `incomplete_*` are **transient network retry cases**, not gaps — resolve query issues rather than mis-triggering AlphaGenome.
- **Layer 5 (Bayesian Evidence Fusion):** Fuses observations with `use_in_fusion=True` into a unified regulatory score. Keep `gnomad_af` out of the fused posterior (`use_in_fusion=False`).
- **Layer 6 (Calibration & Stratification):** Before calibrating against ClinVar reference benchmarks, check label balance per evidence stratum (common variants are evidence-rich and predominantly benign, whereas rare variants are evidence-sparse and pathogenic-enriched).
- **Layer 7 (Output):** Final prioritized variant set.